# Battery Energy Storage System (BESS) Optimization - Synthetic Data Generator

## Overview
This notebook generates synthetic data for battery storage optimization:
- **Battery telemetry** - SOC, voltage, current, temperature
- **Charge/discharge cycles** - Historical cycling data
- **Market prices** - Real-time and day-ahead LMP
- **Grid signals** - Frequency, demand, renewable curtailment

## Tables Created
| Table | Description | Volume |
|-------|-------------|--------|
| `dim_bess_systems` | Battery storage installations | ~30 rows |
| `dim_battery_modules` | Individual battery modules | ~500 rows |
| `fact_bms_telemetry` | Battery management readings | 500K+ rows/day |
| `fact_charge_cycles` | Charge/discharge events | ~5K rows/day |
| `fact_market_signals` | Grid and market signals | ~50K rows/day |
| `fact_dispatch_commands` | Optimization dispatch commands | ~2K rows/day |

## How to Run in Fabric
1. Attach this notebook to a Lakehouse
2. Run all cells sequentially
3. Data will be written as Delta tables

In [ ]:
!pip install faker

In [ ]:
# Configuration
SEED = 42
NUM_BESS_SYSTEMS = 30
MODULES_PER_SYSTEM = 16

from datetime import datetime, timedelta
START_DATE = datetime(2024, 7, 1)  # Summer peak demand
END_DATE = datetime(2024, 7, 8)

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
from datetime import datetime, timedelta
import random
import uuid

np.random.seed(SEED)
random.seed(SEED)
fake = Faker()
Faker.seed(SEED)

print("Libraries loaded successfully")

## 1. Generate BESS System Dimension

In [ ]:
def generate_bess_systems(num_systems):
    systems = []
    
    chemistries = [
        ('LFP', 3.2, 6000, 0.95),  # LiFePO4: voltage, cycles, efficiency
        ('NMC', 3.7, 4000, 0.94),  # Lithium Nickel Manganese Cobalt
        ('NCA', 3.6, 3500, 0.93),  # Lithium Nickel Cobalt Aluminum
    ]
    
    for i in range(num_systems):
        chemistry, nom_voltage, cycle_life, efficiency = random.choice(chemistries)
        power_mw = random.choice([10, 25, 50, 100, 200])
        duration_hours = random.choice([2, 4, 6])
        
        systems.append({
            'system_id': f'BESS-{str(i+1).zfill(3)}',
            'system_name': f'{fake.city()} Energy Storage',
            'latitude': round(random.uniform(30, 45), 4),
            'longitude': round(random.uniform(-120, -75), 4),
            'chemistry': chemistry,
            'power_capacity_mw': power_mw,
            'energy_capacity_mwh': power_mw * duration_hours,
            'duration_hours': duration_hours,
            'nominal_voltage_v': nom_voltage * 100,  # String voltage
            'rated_cycle_life': cycle_life,
            'round_trip_efficiency': efficiency,
            'install_date': fake.date_between(start_date='-5y', end_date='-6m'),
            'commissioning_date': fake.date_between(start_date='-5y', end_date='-6m'),
            'manufacturer': random.choice(['Tesla', 'BYD', 'LG Energy', 'CATL', 'Fluence']),
            'grid_connection': f'SUB-{random.randint(100, 999)}',
            'application': random.choices(
                ['Arbitrage', 'Frequency Regulation', 'Peak Shaving', 'Renewable Integration'],
                weights=[0.3, 0.3, 0.25, 0.15]
            )[0]
        })
    
    return pd.DataFrame(systems)

df_bess_systems = generate_bess_systems(NUM_BESS_SYSTEMS)
print(f"Generated {len(df_bess_systems)} BESS systems")
print(f"Total power: {df_bess_systems['power_capacity_mw'].sum()} MW")
print(f"Total energy: {df_bess_systems['energy_capacity_mwh'].sum()} MWh")
df_bess_systems.head()

In [ ]:
def generate_battery_modules(bess_systems, modules_per_system):
    modules = []
    
    for _, system in bess_systems.iterrows():
        for m in range(modules_per_system):
            # Each module is a portion of total capacity
            module_energy = system['energy_capacity_mwh'] / modules_per_system
            
            # SOH degrades over time
            age_years = (datetime.now() - pd.to_datetime(system['install_date'])).days / 365
            soh = max(0.7, 1 - (age_years * 0.03) - np.random.uniform(0, 0.05))
            
            modules.append({
                'module_id': f'{system["system_id"]}-MOD-{str(m+1).zfill(2)}',
                'system_id': system['system_id'],
                'rack_number': (m // 4) + 1,
                'position_in_rack': (m % 4) + 1,
                'nominal_capacity_kwh': round(module_energy * 1000, 1),
                'usable_capacity_kwh': round(module_energy * 1000 * soh, 1),
                'state_of_health_pct': round(soh * 100, 1),
                'cycle_count': int(age_years * 365 * random.uniform(0.5, 1.5)),
                'manufacture_date': fake.date_between(start_date=system['install_date'] - timedelta(days=180),
                                                      end_date=system['install_date']),
                'serial_number': fake.uuid4()[:16].upper()
            })
    
    return pd.DataFrame(modules)

df_modules = generate_battery_modules(df_bess_systems, MODULES_PER_SYSTEM)
print(f"Generated {len(df_modules)} battery modules")
print(f"Avg SOH: {df_modules['state_of_health_pct'].mean():.1f}%")
df_modules.head()

## 2. Generate BMS Telemetry Data

In [ ]:
def generate_bms_telemetry(bess_systems, modules_df, start_time, hours=24):
    """Generate battery management system telemetry"""
    telemetry = []
    
    for _, system in bess_systems.iterrows():
        system_modules = modules_df[modules_df['system_id'] == system['system_id']]
        
        # Initialize SOC randomly
        current_soc = random.uniform(0.3, 0.7)
        
        current_time = start_time
        while current_time < start_time + timedelta(hours=hours):
            hour = current_time.hour
            
            # Simulate charge/discharge based on time of day
            if 10 <= hour <= 14:  # Charge during solar hours
                power_flow = random.uniform(0.3, 0.8)  # Charging (positive)
                current_soc = min(0.95, current_soc + 0.02)
            elif 17 <= hour <= 21:  # Discharge during evening peak
                power_flow = random.uniform(-0.8, -0.3)  # Discharging (negative)
                current_soc = max(0.1, current_soc - 0.03)
            else:
                power_flow = random.uniform(-0.2, 0.2)  # Idle/regulation
                current_soc += power_flow * 0.005
            
            for _, module in system_modules.iterrows():
                # Module-level variations
                module_soc = current_soc + np.random.normal(0, 0.02)
                module_soc = np.clip(module_soc, 0.05, 0.98)
                
                # Temperature increases during charge/discharge
                ambient_temp = 25 + 5 * np.sin((hour - 6) * np.pi / 12)
                temp_rise = abs(power_flow) * 10
                cell_temp = ambient_temp + temp_rise + np.random.normal(0, 2)
                
                # Voltage based on SOC (simplified)
                cell_voltage = system['nominal_voltage_v'] * (0.85 + 0.15 * module_soc)
                
                # Current based on power flow
                current = power_flow * system['power_capacity_mw'] * 1000 / cell_voltage
                
                # Cell imbalance detection
                has_imbalance = random.random() < 0.02
                
                telemetry.append({
                    'telemetry_id': str(uuid.uuid4()),
                    'system_id': system['system_id'],
                    'module_id': module['module_id'],
                    'timestamp': current_time,
                    'soc_pct': round(module_soc * 100, 1),
                    'voltage_v': round(cell_voltage, 1),
                    'current_a': round(current, 1),
                    'power_kw': round(cell_voltage * current / 1000, 1),
                    'cell_temp_max_c': round(cell_temp + np.random.uniform(0, 3), 1),
                    'cell_temp_min_c': round(cell_temp - np.random.uniform(0, 3), 1),
                    'cell_temp_avg_c': round(cell_temp, 1),
                    'cell_voltage_max_v': round(cell_voltage * 1.02 / 100, 3),
                    'cell_voltage_min_v': round(cell_voltage * 0.98 / 100, 3),
                    'cell_imbalance_detected': has_imbalance,
                    'ambient_temp_c': round(ambient_temp, 1)
                })
            
            current_time += timedelta(minutes=5)
    
    return pd.DataFrame(telemetry)

# Generate for first 5 systems to keep manageable size
df_telemetry = generate_bms_telemetry(df_bess_systems.head(5), df_modules, START_DATE, hours=24)
print(f"Generated {len(df_telemetry)} BMS telemetry records")
print(f"SOC range: {df_telemetry['soc_pct'].min():.1f}% - {df_telemetry['soc_pct'].max():.1f}%")
df_telemetry.head()

## 3. Generate Charge/Discharge Cycles

In [ ]:
def generate_charge_cycles(bess_systems, start_date, days=7):
    """Generate charge and discharge cycle events"""
    cycles = []
    
    for _, system in bess_systems.iterrows():
        current_time = start_date
        
        while current_time < start_date + timedelta(days=days):
            hour = current_time.hour
            
            # Morning charge cycle (solar)
            if hour == 10:
                start_soc = random.uniform(15, 30)
                end_soc = random.uniform(85, 95)
                duration = timedelta(hours=random.uniform(3, 5))
                
                cycles.append({
                    'cycle_id': str(uuid.uuid4()),
                    'system_id': system['system_id'],
                    'cycle_type': 'Charge',
                    'start_timestamp': current_time,
                    'end_timestamp': current_time + duration,
                    'start_soc_pct': round(start_soc, 1),
                    'end_soc_pct': round(end_soc, 1),
                    'energy_mwh': round(system['energy_capacity_mwh'] * (end_soc - start_soc) / 100, 1),
                    'avg_power_mw': round(system['power_capacity_mw'] * random.uniform(0.5, 0.8), 1),
                    'avg_c_rate': round(random.uniform(0.2, 0.5), 2),
                    'efficiency_pct': round(random.uniform(93, 97), 1),
                    'max_temp_c': round(random.uniform(35, 45), 1),
                    'trigger': 'Price Signal'
                })
            
            # Evening discharge cycle (peak)
            if hour == 17:
                start_soc = random.uniform(85, 95)
                end_soc = random.uniform(15, 25)
                duration = timedelta(hours=random.uniform(3, 5))
                
                cycles.append({
                    'cycle_id': str(uuid.uuid4()),
                    'system_id': system['system_id'],
                    'cycle_type': 'Discharge',
                    'start_timestamp': current_time,
                    'end_timestamp': current_time + duration,
                    'start_soc_pct': round(start_soc, 1),
                    'end_soc_pct': round(end_soc, 1),
                    'energy_mwh': round(system['energy_capacity_mwh'] * (start_soc - end_soc) / 100, 1),
                    'avg_power_mw': round(system['power_capacity_mw'] * random.uniform(0.6, 0.9), 1),
                    'avg_c_rate': round(random.uniform(0.3, 0.5), 2),
                    'efficiency_pct': round(random.uniform(94, 98), 1),
                    'max_temp_c': round(random.uniform(38, 48), 1),
                    'trigger': 'Peak Demand'
                })
            
            current_time += timedelta(hours=1)
    
    return pd.DataFrame(cycles)

df_cycles = generate_charge_cycles(df_bess_systems, START_DATE, days=7)
print(f"Generated {len(df_cycles)} charge/discharge cycles")
print(f"Total energy cycled: {df_cycles['energy_mwh'].sum():,.0f} MWh")
df_cycles.head()

## 4. Generate Market and Grid Signals

In [ ]:
def generate_market_signals(start_date, days=7):
    """Generate grid and market signals for optimization"""
    signals = []
    
    current_time = start_date
    while current_time < start_date + timedelta(days=days):
        hour = current_time.hour
        minute = current_time.minute
        is_weekend = current_time.weekday() >= 5
        
        # Grid frequency (nominal 60 Hz)
        frequency = 60.0 + np.random.normal(0, 0.01)
        
        # LMP prices (similar to renewable notebook)
        if is_weekend:
            base_price = 30 + 15 * np.sin((hour - 6) * np.pi / 12)
        else:
            if 7 <= hour <= 20:
                base_price = 50 + 30 * np.sin((hour - 6) * np.pi / 12)
            else:
                base_price = 25
        
        lmp = base_price + np.random.normal(0, 5)
        
        # Price spikes
        if random.random() < 0.02:
            lmp *= random.uniform(2, 4)
        
        # Negative prices during solar peak
        if random.random() < 0.03 and 10 <= hour <= 14:
            lmp = -random.uniform(5, 25)
        
        # Day-ahead price (forecast for same hour tomorrow)
        da_price = lmp * random.uniform(0.9, 1.1)
        
        # Grid demand (MW)
        base_demand = 50000 if not is_weekend else 40000
        demand = base_demand + 15000 * np.sin((hour - 6) * np.pi / 12) + np.random.normal(0, 2000)
        
        # Renewable curtailment signal
        curtailment_risk = 0
        if 10 <= hour <= 14 and lmp < 0:
            curtailment_risk = random.uniform(0.3, 0.8)
        
        # Regulation signal (AGC)
        reg_signal = np.random.normal(0, 0.3)  # -1 to 1 range
        
        signals.append({
            'signal_id': str(uuid.uuid4()),
            'timestamp': current_time,
            'grid_frequency_hz': round(frequency, 4),
            'lmp_realtime_usd': round(lmp, 2),
            'lmp_dayahead_usd': round(da_price, 2),
            'price_spread_usd': round(da_price - lmp, 2),
            'grid_demand_mw': round(demand, 0),
            'renewable_curtailment_risk': round(curtailment_risk, 2),
            'regulation_signal': round(np.clip(reg_signal, -1, 1), 3),
            'reserve_requirement_mw': round(demand * 0.03, 0),
            'carbon_intensity_kg_mwh': round(max(100, 400 - 200 * curtailment_risk + np.random.normal(0, 30)), 0)
        })
        
        current_time += timedelta(minutes=5)
    
    return pd.DataFrame(signals)

df_signals = generate_market_signals(START_DATE, days=7)
print(f"Generated {len(df_signals)} market signal records")
print(f"Price range: ${df_signals['lmp_realtime_usd'].min():.2f} to ${df_signals['lmp_realtime_usd'].max():.2f}")
df_signals.head()

## 5. Generate Dispatch Commands

In [ ]:
def generate_dispatch_commands(bess_systems, signals_df, start_date, days=7):
    """Generate optimization dispatch commands"""
    commands = []
    
    for _, system in bess_systems.iterrows():
        for _, signal in signals_df.iterrows():
            hour = signal['timestamp'].hour
            
            # Determine dispatch action based on price and signals
            if signal['lmp_realtime_usd'] < 0:
                action = 'CHARGE'
                power_setpoint = system['power_capacity_mw'] * random.uniform(0.7, 1.0)
                reason = 'Negative Price Opportunity'
            elif signal['lmp_realtime_usd'] > 80:
                action = 'DISCHARGE'
                power_setpoint = -system['power_capacity_mw'] * random.uniform(0.8, 1.0)
                reason = 'High Price Opportunity'
            elif abs(signal['regulation_signal']) > 0.5:
                action = 'REGULATE'
                power_setpoint = system['power_capacity_mw'] * signal['regulation_signal'] * 0.3
                reason = 'Frequency Regulation'
            elif signal['renewable_curtailment_risk'] > 0.5:
                action = 'CHARGE'
                power_setpoint = system['power_capacity_mw'] * random.uniform(0.5, 0.8)
                reason = 'Curtailment Avoidance'
            else:
                action = 'IDLE'
                power_setpoint = 0
                reason = 'Standby'
            
            # Only log non-idle commands or sampled idle
            if action != 'IDLE' or random.random() < 0.1:
                commands.append({
                    'command_id': str(uuid.uuid4()),
                    'system_id': system['system_id'],
                    'timestamp': signal['timestamp'],
                    'action': action,
                    'power_setpoint_mw': round(power_setpoint, 2),
                    'reason': reason,
                    'lmp_at_dispatch': signal['lmp_realtime_usd'],
                    'expected_revenue_usd': round(abs(power_setpoint) * signal['lmp_realtime_usd'] / 12, 2),
                    'command_source': random.choice(['Optimizer', 'AGC', 'Manual']),
                    'executed': random.random() > 0.02  # 98% success rate
                })
    
    return pd.DataFrame(commands)

# Use first 10 systems to keep size manageable
df_commands = generate_dispatch_commands(df_bess_systems.head(10), df_signals, START_DATE)
print(f"Generated {len(df_commands)} dispatch commands")
print(f"Actions: {df_commands['action'].value_counts().to_dict()}")
df_commands.head()

## 6. Visualize BESS Operations

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# SOC over time for first system
system1_telemetry = df_telemetry[df_telemetry['system_id'] == df_bess_systems.iloc[0]['system_id']]
soc_avg = system1_telemetry.groupby('timestamp')['soc_pct'].mean()
axes[0].plot(soc_avg.index, soc_avg.values, color='green', linewidth=1)
axes[0].axhline(y=90, color='red', linestyle='--', alpha=0.5, label='Max SOC')
axes[0].axhline(y=10, color='red', linestyle='--', alpha=0.5, label='Min SOC')
axes[0].set_title(f'State of Charge - {df_bess_systems.iloc[0]["system_name"]}')
axes[0].set_ylabel('SOC (%)')
axes[0].set_ylim(0, 100)
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Market prices
axes[1].plot(df_signals['timestamp'], df_signals['lmp_realtime_usd'], color='blue', linewidth=1, label='Real-Time')
axes[1].plot(df_signals['timestamp'], df_signals['lmp_dayahead_usd'], color='orange', linewidth=1, alpha=0.7, label='Day-Ahead')
axes[1].axhline(y=0, color='green', linestyle='--', alpha=0.5)
axes[1].set_title('Market Prices')
axes[1].set_ylabel('$/MWh')
axes[1].grid(True, alpha=0.3)
axes[1].legend()

# Cycle counts by type
cycle_counts = df_cycles.groupby(['cycle_type', df_cycles['start_timestamp'].dt.date]).size().unstack(level=0)
cycle_counts.plot(kind='bar', ax=axes[2], color=['blue', 'orange'])
axes[2].set_title('Daily Charge/Discharge Cycles')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 7. Data Validation

In [ ]:
print("=== Data Validation Report ===")
print(f"\nBESS Systems: {len(df_bess_systems)} records")
print(f"  - Total power: {df_bess_systems['power_capacity_mw'].sum()} MW")
print(f"  - Total energy: {df_bess_systems['energy_capacity_mwh'].sum()} MWh")

print(f"\nBattery Modules: {len(df_modules)} records")
print(f"  - Avg SOH: {df_modules['state_of_health_pct'].mean():.1f}%")
print(f"  - Valid FK: {df_modules['system_id'].isin(df_bess_systems['system_id']).all()}")

print(f"\nBMS Telemetry: {len(df_telemetry)} records")
print(f"  - SOC range: {df_telemetry['soc_pct'].min():.1f}% - {df_telemetry['soc_pct'].max():.1f}%")
print(f"  - Temp range: {df_telemetry['cell_temp_avg_c'].min():.1f}°C - {df_telemetry['cell_temp_avg_c'].max():.1f}°C")

print(f"\nCharge Cycles: {len(df_cycles)} records")
print(f"  - Total energy: {df_cycles['energy_mwh'].sum():,.0f} MWh")

print(f"\nMarket Signals: {len(df_signals)} records")
print(f"  - Price range: ${df_signals['lmp_realtime_usd'].min():.2f} to ${df_signals['lmp_realtime_usd'].max():.2f}")
print(f"  - Negative prices: {len(df_signals[df_signals['lmp_realtime_usd'] < 0])}")

print(f"\nDispatch Commands: {len(df_commands)} records")
print(f"  - Execution rate: {df_commands['executed'].mean() * 100:.1f}%")

## 8. Write to Lakehouse

In [ ]:
# Uncomment for Fabric execution
# from pyspark.sql import SparkSession
# spark = SparkSession.builder.getOrCreate()

# spark.createDataFrame(df_bess_systems).write.mode("overwrite").format("delta").saveAsTable("dim_bess_systems")
# spark.createDataFrame(df_modules).write.mode("overwrite").format("delta").saveAsTable("dim_battery_modules")
# spark.createDataFrame(df_telemetry).write.mode("append").format("delta").saveAsTable("fact_bms_telemetry")
# spark.createDataFrame(df_cycles).write.mode("append").format("delta").saveAsTable("fact_charge_cycles")
# spark.createDataFrame(df_signals).write.mode("append").format("delta").saveAsTable("fact_market_signals")
# spark.createDataFrame(df_commands).write.mode("append").format("delta").saveAsTable("fact_dispatch_commands")

# Local testing
df_bess_systems.to_csv('dim_bess_systems.csv', index=False)
df_telemetry.to_csv('fact_bms_telemetry.csv', index=False)
print("Data saved successfully")

## Known Limitations
1. SOC transitions are simplified - real BMS has complex cell balancing
2. Degradation model is linear - actual capacity fade follows non-linear curves
3. Market signals don't reflect real ISO bid/offer structures

## Next Steps
1. Build optimization model for arbitrage decisions
2. Train SOH prediction model from cycling data
3. Implement real-time dispatch with Activator integration